In [ ]:
import pandas as pd
import re

# Logic to clean the data manually
cleaned_data = []
current_id = None
current_text = []

# Regex to find the "Anchor" (Start of a new row)
# Looks for: Start of line (^), Digits (\d+), followed by a comma (,)
new_row_pattern = re.compile(r'^(\d+),(.*)')

print("Starting manual parsing... this handles broken newlines and commas.")

with open('/content/drive/MyDrive/Steam/reviews only.csv', 'r', encoding='utf-8', errors='replace') as f:
    for i, line in enumerate(f):
        line = line.strip() # Remove trailing newline characters

        # Check if this line is the start of a new row (ID, Text...)
        match = new_row_pattern.match(line)

        if match:
            # 1. SAVE PREVIOUS: If we were building a row, save it now
            if current_id is not None:
                full_review = " ".join(current_text) # Merge multi-line text
                cleaned_data.append({'recommendationid': current_id, 'review_text': full_review})

            # 2. START NEW: Start the new row
            current_id = match.group(1)      # The ID (e.g., 10000000)
            initial_text = match.group(2)    # The start of the review
            current_text = [initial_text]    # Start the text buffer

        else:
            # 3. APPEND: This line is a continuation of the previous review
            # (Only if we have a valid ID started)
            if current_id is not None:
                current_text.append(line)

    # Don't forget to save the very last row!
    if current_id is not None:
        full_review = " ".join(current_text)
        cleaned_data.append({'recommendationid': current_id, 'review_text': full_review})

# Convert to DataFrame
df_clean = pd.DataFrame(cleaned_data)

print(f"Successfully reconstructed {len(df_clean)} rows.")


Starting manual parsing... this handles broken newlines and commas.
Successfully reconstructed 1048674 rows.


In [ ]:
print(df_clean.head(10))

  recommendationid                                        review_text
0         10000000  "What's a crap. This game costs 2 euro but is ...
1        100001066  "Игра в жанре квеста point-&-click, но особенн...
2        100002344  Erneut gibt es einen DLC mit drei neuen Lords ...
3        100002361  "It took me a grueling 18 months, but I built ...
4        100002504  "incredibly simple game. it does get boring qu...
5        100002591  "Un juego entretenido que he regalado a mis so...
6        100003580  "一言で言うと2か所のチェックポイントを通過し、ゴールを目指すゲーム。 なんでライフルを持っ...
7        100003676  "If you enjoy text-based games, this is a must...
8        100003804                                         very cool.
9        100005000                                             "ДИЧЬ"


In [ ]:
demo = df_clean.head(100).copy()

In [ ]:
import pandas as pd
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset
import torch
from tqdm.auto import tqdm
import os
import time

In [ ]:
# --- CONFIGURATION ---
drive_path = '/content/drive/MyDrive/Steam/reviews_scored_partial.csv'
batch_save_size = 50000   # Save to Drive every 50k
status_update_interval = 10000 # Print status every 10k

In [ ]:
"""
import pandas as pd
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset
import torch
from tqdm.auto import tqdm
import os

# --- STEP 1: PREPARATION ---

# Check for GPU (Essential for 1M rows)
device = 0 if torch.cuda.is_available() else -1
print(f"Using Device: {'GPU (Fast)' if device == 0 else 'CPU (Slow)'}")

# Assume df_clean is your dataframe from the previous step
# 1. Clean the quotes we saw in the preview
df_clean['review_text'] = df_clean['review_text'].str.strip('"').str.strip()

# 2. Setup Model
model_name = 'nlptown/bert-base-multilingual-uncased-sentiment'
sentiment_pipeline = pipeline(
    task='sentiment-analysis',
    model=model_name,
    device=device,
    truncation=True,
    max_length=512,
    batch_size=64
)

# --- STEP 2: BATCH PROCESSING WITH AUTOSAVE ---

results = []
batch_save_size = 50000  # Save progress every 50k rows
output_file = 'reviews_scored_partial.csv'

# Define the mapping
star_map = {1: 'Worse', 2: 'Bad', 3: 'Good', 4: 'Better', 5: 'Best'}

print(f"Starting processing on {len(demo)} rows...")

# We iterate through the pipeline generator
for i, out in tqdm(enumerate(sentiment_pipeline(KeyDataset(demo.to_dict('records'), "review_text"))), total=len(df_clean)):

    # Extract Score (e.g., '4 stars' -> 4)
    star = int(out['label'].split()[0])
    category = star_map.get(star, 'Unknown')

    results.append({
        'recommendationid': df_clean.iloc[i]['recommendationid'],
        'numeric_score': star,
        'category': category
    })

    # SAFETY SAVE: Every 50k rows, write to disk
    if (i + 1) % batch_save_size == 0:
        pd.DataFrame(results).to_csv(output_file, index=False)
        print(f"  [Safety Checkpoint] Saved {i+1} rows to {output_file}")

# --- STEP 3: FINAL MERGE ---

print("Processing Complete. Creating final file.")
final_df = pd.DataFrame(results)

# Merge back with original text if you want the text in the final file
# (Optional: might make the file huge, 500MB+)
full_df = df_clean.merge(demo, on='recommendationid')

full_df.to_csv('reviews_final_scored.csv', index=False)
print("Saved to reviews_final_scored.csv")

"""

Using Device: GPU (Fast)


Device set to use cuda:0


Starting processing on 100 rows...


  0%|          | 0/1048674 [00:00<?, ?it/s]

Processing Complete. Creating final file.
Saved to reviews_final_scored.csv


In [ ]:
import pandas as pd
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset
import torch
from tqdm.auto import tqdm
import os
import time

# ==========================================
# CONFIGURATION
# ==========================================
drive_path = '/content/drive/MyDrive/Steam/reviews_scored_final.csv'
batch_save_size = 50000
status_interval = 10000

# ==========================================
# 1. RESET BAD FILE
# ==========================================
if os.path.exists(drive_path):
    print(f"⚠️ Deleting bad file: {drive_path}")
    os.remove(drive_path)

# Create fresh file
pd.DataFrame(columns=['recommendationid', 'numeric_score', 'category']).to_csv(drive_path, index=False)
print("✅ Created fresh output file.")

# Check if df_clean exists (from previous cell)
if 'df_clean' not in locals():
    raise ValueError("df_clean is missing! Please run the 'Data Cleaning' cell again.")

# ==========================================
# 2. SETUP MODEL
# ==========================================
device = 0 if torch.cuda.is_available() else -1
print(f"Using Device: {'GPU 🚀' if device == 0 else 'CPU 🐢'}")

sentiment_pipeline = pipeline(
    task='sentiment-analysis',
    model='nlptown/bert-base-multilingual-uncased-sentiment',
    device=device,
    truncation=True,
    max_length=512,
    batch_size=32 # Optimized for T4
)

star_map = {1: 'Worse', 2: 'Bad', 3: 'Neutral', 4: 'Good', 5: 'Best'}

# ==========================================
# 3. CORRECT PROCESSING LOOP
# ==========================================
print(f"Processing {len(df_clean)} rows...")

# Initialize data stream
data_stream = KeyDataset(df_clean.to_dict('records'), "review_text")

current_batch = []
start_time = time.time()

print("-" * 50)

# --- THE FIX IS HERE ---
# We iterate over the PIPELINE result, not the data_stream itself
pipeline_iterator = sentiment_pipeline(data_stream, batch_size=32)

for i, out in tqdm(enumerate(pipeline_iterator), total=len(df_clean)):

    # DEBUG: Print first row to prove it works this time
    if i == 0:
        print(f"\n[DEBUG CHECK] Row 0 Output: {out}")
        # Should be: {'label': '1 star', 'score': ...}

    try:
        # Handle Output
        if isinstance(out, list):
            result = out[0]
        else:
            result = out

        star = int(result['label'].split()[0])
        category = star_map.get(star, 'Unknown')

    except Exception as e:
        if i < 5: print(f"❌ Error on row {i}: {e}")
        star = -1
        category = "Error"

    # Save Result
    original_idx = df_clean.index[i]
    current_batch.append({
        'recommendationid': df_clean.at[original_idx, 'recommendationid'],
        'numeric_score': star,
        'category': category
    })

    # STATUS & SAVE
    if (i + 1) % status_interval == 0:
        elapsed = time.time() - start_time
        print(f"Status: {i + 1} rows done. (Last 10k took: {elapsed:.2f}s)")
        start_time = time.time()

    if (i + 1) % batch_save_size == 0:
        pd.DataFrame(current_batch).to_csv(drive_path, mode='a', header=False, index=False)
        print(f"  💾 SAVED to Drive: {i + 1} rows safe.")
        current_batch = []

# Final Save
if current_batch:
    pd.DataFrame(current_batch).to_csv(drive_path, mode='a', header=False, index=False)
    print("  💾 Final Batch Saved.")

print(f"\n✅ COMPLETE! File saved at: {drive_path}")

⚠️ Deleting bad file: /content/drive/MyDrive/Steam/reviews_scored_final.csv
✅ Created fresh output file.
Using Device: GPU 🚀


Device set to use cuda:0


Processing 1048674 rows...
--------------------------------------------------


  0%|          | 0/1048674 [00:00<?, ?it/s]


[DEBUG CHECK] Row 0 Output: {'label': '1 star', 'score': 0.9680441617965698}
Status: 10000 rows done. (Last 10k took: 254.73s)
Status: 20000 rows done. (Last 10k took: 252.60s)
Status: 30000 rows done. (Last 10k took: 257.47s)
Status: 40000 rows done. (Last 10k took: 258.02s)
Status: 50000 rows done. (Last 10k took: 262.80s)
  💾 SAVED to Drive: 50000 rows safe.
Status: 60000 rows done. (Last 10k took: 258.86s)
Status: 70000 rows done. (Last 10k took: 267.72s)
Status: 80000 rows done. (Last 10k took: 270.81s)
Status: 90000 rows done. (Last 10k took: 261.07s)
Status: 100000 rows done. (Last 10k took: 266.10s)
  💾 SAVED to Drive: 100000 rows safe.
Status: 110000 rows done. (Last 10k took: 268.18s)
Status: 120000 rows done. (Last 10k took: 268.53s)
Status: 130000 rows done. (Last 10k took: 266.02s)
Status: 140000 rows done. (Last 10k took: 265.52s)
Status: 150000 rows done. (Last 10k took: 253.88s)
  💾 SAVED to Drive: 150000 rows safe.
Status: 160000 rows done. (Last 10k took: 258.82s)
St

In [ ]:
res1 = pd.read_csv("/content/drive/MyDrive/Steam/reviews_scored_final.csv")
res1.head(10)

,recommendationid,numeric_score,category
0,10000000,1,Worse
1,100001066,3,Good
2,100002344,4,Better
3,100002361,5,Best
4,100002504,3,Good
5,100002591,4,Better
6,100003580,4,Better
7,100003676,4,Better
8,100003804,5,Best
9,100005000,3,Good
